### 1. 패키지 설치 + 환경변수 로드

In [1]:
%pip install -qU langchain langchain_openai langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. interrupt 개요

노드 실행 도중 그래프를 멈추고 사람의 입력을 기다리는 기능임

**전제 조건**
- `checkpointer` — 멈춘 시점의 State를 저장해야 재개할 수 있음
- `thread_id` — 어떤 세션을 재개할지 식별

**주의사항** (공식 문서)
- 노드 하나당 `interrupt()` 는 **정확히 한 번**만 호출
- `interrupt()` 를 `try/except` 로 감싸지 말 것 (멈춤 신호를 삼킴)
- 재개 시 **노드가 처음부터 다시 실행**되므로, 부작용이 있는 작업은 별도 노드로 분리
- JSON 직렬화 가능한 값만 전달

### 3. 승인 절차가 있는 그래프

In [3]:
from typing_extensions import TypedDict, NotRequired
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt

class State(TypedDict):
    action: str
    approved: NotRequired[bool]   # 실행 시점에 채워지는 값이므로 선택 키
    result: NotRequired[str]

def review(state: State):
    approved = interrupt(f"'{state['action']}' 을(를) 실행할까요?")   # 여기서 그래프가 멈춤
    return {"approved": approved}

def execute(state: State):
    if state.get("approved"):   # 선택 키이므로 get 으로 접근
        return {"result": f"{state['action']} 실행 완료"}
    return {"result": f"{state['action']} 취소됨"}

graph_builder = StateGraph(State)
graph_builder.add_node("review", review)
graph_builder.add_node("execute", execute)   # 부작용이 있는 작업은 interrupt 노드와 분리
graph_builder.add_edge(START, "review")
graph_builder.add_edge("review", "execute")
graph_builder.add_edge("execute", END)

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

### 4. 실행 → 멈춤

`invoke()` 결과의 `__interrupt__` 키에 멈춘 이유가 담김

In [4]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

result = graph.invoke({"action": "파일 삭제"}, config)
print(result["__interrupt__"])

[Interrupt(value="'파일 삭제' 을(를) 실행할까요?", id='fa9f51b0df2bee0be0a3b6a7726ae079')]


### 5. 멈춘 지점 확인

`next` 를 보면 어느 노드에서 대기 중인지 알 수 있음

In [5]:
snapshot = graph.get_state(config)
print("다음 실행 노드:", snapshot.next)
print("현재 State:", snapshot.values)

다음 실행 노드: ('review',)
현재 State: {'action': '파일 삭제'}


### 6. 승인하여 재개

`Command(resume=...)` 로 사람의 답을 주입함

In [6]:
from langgraph.types import Command

result = graph.invoke(Command(resume=True), config)
print(result["result"])

파일 삭제 실행 완료


### 7. 거부하는 경우

새 thread에서 같은 흐름을 `resume=False` 로 진행함

In [7]:
config_2: RunnableConfig = {"configurable": {"thread_id": "2"}}

graph.invoke({"action": "파일 삭제"}, config_2)   # 멈춤
result = graph.invoke(Command(resume=False), config_2)
print(result["result"])

파일 삭제 취소됨


### 8. 정적 중단점 (디버깅용)

`interrupt()` 없이도 컴파일 시점에 특정 노드 앞뒤에서 멈출 수 있음. 재개는 입력 `None` 으로 함

In [8]:
debug_graph = graph_builder.compile(
    checkpointer=InMemorySaver(),
    interrupt_before=["execute"],   # execute 실행 직전에 멈춤
)

config_3: RunnableConfig = {"configurable": {"thread_id": "3"}}

debug_graph.invoke({"action": "파일 삭제"}, config_3)              # review의 interrupt에서 멈춤
debug_graph.invoke(Command(resume=True), config_3)                 # execute 직전에서 다시 멈춤
print("멈춘 지점:", debug_graph.get_state(config_3).next)
print(debug_graph.invoke(None, config_3)["result"])                # None으로 재개

멈춘 지점: ('execute',)
파일 삭제 실행 완료


### 9. 정리

- `interrupt()` 는 노드 안에서 사람의 입력을 요청하고, `Command(resume=...)` 로 재개함
- 재개 시 **노드 전체가 처음부터 다시 실행**되므로 부작용 작업은 별도 노드로 분리함
- `interrupt_before` / `interrupt_after` 는 디버깅용 정적 중단점이며 `None` 입력으로 재개함
- 참고: [Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)